# Azure OpenAI Decisions API Hands-on

## サマリー

テキストや画像から、文章ではなく確率・選択肢・スコアを取得します。

| Part | 内容 |
| --- | --- |
| 0 | インストール、接続設定、クライアント、グラフの準備 |
| 1 | `predicate`、`choice`、`score`、スコア計算、レスポンス、REST |
| 2 | 複数質問、メッセージ入力、画像の準備、画像判定、複数画像 |

- Azure OpenAI リソースと `gpt-6-luna` のデプロイが必要です。`model` にはデプロイ名を指定します。
- 上から順に実行してください。後のセルは、前のセルで定義した質問や設定を使用します。
- 判定結果は JSON と確率分布のグラフで確認できます。描画には取得済みの結果を使い、追加の API 呼び出しは行いません。
- 全セルを 1 回ずつ実行すると、モデルへのリクエストは計 8 回です。各呼び出しは課金対象です。テキストだけを試す場合は 2-2 で終了できます。
- 判定値はモデルの推定です。日本語での精度や特定の数値を保証するものではありません。

出典: [Decisions API](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions)、[料金](https://azure.microsoft.com/pricing/details/azure-openai/)

## 全体像: 作家ではなく審査員

Chat Completions や Responses API は、文章を書く作家のような API です。Decisions API は、渡された材料を見て札を上げる審査員のような API です。返ってくるのは文章ではなく、アプリケーションがそのまま分岐に使える値です。

| 質問タイプ | たとえ | 返ってくる値 |
| --- | --- | --- |
| `predicate` | 降水確率。明日は雨が降るか | `probability`（0 から 1 の確率） |
| `choice` | 郵便物の仕分け箱。どの箱に入れるか | `choice`（渡した値のどれか）と各選択肢の確率 |
| `score` | 辛さの目盛り。どの段階か | `score`（段階の添字の確率加重平均）と各段階の確率 |

出典: 「Understand how decisions work」 https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#understand-how-decisions-work

リクエストは `model`、`input`、`questions` の 3 つで構成されます。レスポンスの `answers` は `questions` と同じ順序で並び、各回答には質問の `name` が入ります。

```mermaid
flowchart LR
    APP[アプリケーション] -->|model, input, questions| EP[POST openai/v1/decisions]
    EP --> ANS[answers 配列<br/>質問と同じ順序]
    EP --> USE[usage<br/>トークン使用量]
    ANS --> P[predicate: probability]
    ANS --> C[choice: choice と probabilities]
    ANS --> S[score: score と probabilities]
    ANS --> R[refusal: name のみ]
```

| フィールド | 役割 |
| --- | --- |
| `model` | `gpt-6-luna` のデプロイ名 |
| `input` | すべての質問が評価する共通の材料。文字列、または user メッセージ（content は文字列、または `input_text` と `input_image` のリスト） |
| `questions` | 評価する内容。種類、指示文、選択肢や段階を持つ |

出典: 同上

## Part 0: セットアップ

### 0-1. 必要なライブラリ

`openai` は公式の前提である 3.26.0 以降を使用します。`httpx` は REST 呼び出し、`azure-identity` は Microsoft Entra ID 認証、`python-dotenv` は既存の `.env` の読み込み、`matplotlib` はグラフの描画に使用します。画像は既存ファイルを使うため、画像生成用のライブラリは不要です。必要であればインストール後にカーネルを再起動してください。

出典: [前提条件](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#prerequisites)

In [ ]:
%pip install -q "openai>=3.26.0" httpx azure-identity python-dotenv matplotlib

### 0-2. 接続設定

両ノートブックで同じ `.env` を使用します。[.env.example](./.env.example) を同じフォルダーに `.env` としてコピーし、接続先と実際のデプロイ名を記入してください。カーネルの作業フォルダーをこのノートブックのあるフォルダーにして実行します。`.env` の値は既存の環境変数を上書きします。変更後は 0-2 と 0-3 を再実行してください。

認証方式と API キーは共通です。接続先とデプロイ名は API ごとに分けているため、同じ `.env` を編集し直さずに両方を実行できます。

| 環境変数 | 使用先 | 内容 |
| --- | --- | --- |
| `USE_ENTRA_ID` | 共通 | 省略時は `true`。API キーを使う場合は `false` |
| `AZURE_API_KEY` | 共通 | API キー認証時のみ使用。未設定なら 0-3 で非表示入力します |
| `AZURE_OPENAI_ENDPOINT` | このノートブック | Azure OpenAI リソースのルート URL。例: `https://YOUR-RESOURCE-NAME.openai.azure.com`。未設定なら入力します |
| `AZURE_OPENAI_DECISIONS_DEPLOYMENT` | このノートブック | デプロイ名。省略時は `gpt-6-luna` |
| `MICROSOFT_DECISION_ENDPOINT` | Microsoft-Decision-1 | Foundry リソースのルート URL。デプロイ詳細ページから取得します |
| `MICROSOFT_DECISION_DEPLOYMENT` | Microsoft-Decision-1 | デプロイ名。省略時は `Microsoft-Decision-1` |

接続先には `/openai/v1/` や `/providers/microsoft/v1/systemone` を含めません。必要なパスは各ノートブックが追加します。API キーを共用できるのは同じリソースの場合だけです。別リソースのキーが必要なら `.env` の `AZURE_API_KEY` を空欄にし、各ノートブックの 0-3 で対応するキーを入力してください。Microsoft Entra ID 認証では、両リソースへの利用権限が必要です。`.env` は Git の管理対象外です。

出典: [エンドポイントとデプロイ名](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#prerequisites)

In [ ]:
import getpass
import json
import os
from urllib.parse import urlsplit
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_ENDPOINT = (os.getenv('AZURE_OPENAI_ENDPOINT') or input('Azure OpenAI リソースのルート URL: ')).strip().rstrip('/')
DEPLOYMENT = os.getenv('AZURE_OPENAI_DECISIONS_DEPLOYMENT', 'gpt-6-luna').strip()
auth_setting = os.getenv('USE_ENTRA_ID', 'true').strip().lower()
if auth_setting not in ('1', 'true', 'yes', '0', 'false', 'no'):
    raise ValueError('USE_ENTRA_ID には true または false を設定してください。')
USE_ENTRA_ID = auth_setting in ('1', 'true', 'yes')
endpoint_parts = urlsplit(AZURE_ENDPOINT)
if (endpoint_parts.scheme != 'https' or not endpoint_parts.hostname
        or endpoint_parts.path or endpoint_parts.query or endpoint_parts.fragment
        or endpoint_parts.username is not None or endpoint_parts.password is not None
        or 'YOUR-RESOURCE-NAME' in AZURE_ENDPOINT or not DEPLOYMENT):
    raise ValueError('AZURE_OPENAI_ENDPOINT に実際のリソースのルート URL、AZURE_OPENAI_DECISIONS_DEPLOYMENT にデプロイ名を設定してください。')
ENDPOINT = AZURE_ENDPOINT + '/openai/v1/'
print('デプロイ:', DEPLOYMENT)

### 0-3. 認証とクライアント

Microsoft Entra ID 認証では、利用権限のあるアカウントで Azure CLI などに事前にサインインしてください。 `DefaultAzureCredential` を使用します。API キーを使う場合も、キー自体は表示しません。自動再試行は無効にしているため、失敗時はエラーを確認してから該当セルを再実行してください。

出典: [認証](https://learn.microsoft.com/azure/ai-services/openai/how-to/managed-identity)

In [ ]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

if USE_ENTRA_ID:
    token_provider = get_bearer_token_provider(DefaultAzureCredential(), 'https://ai.azure.com/.default')
    api_key = token_provider
else:
    API_KEY = os.getenv('AZURE_API_KEY') or getpass.getpass('AZURE_API_KEY: ')
    if not API_KEY:
        raise ValueError('API キーを設定してください。')
    api_key = API_KEY

client = OpenAI(base_url=ENDPOINT, api_key=api_key, timeout=60.0, max_retries=0)
print('認証:', 'Microsoft Entra ID' if USE_ENTRA_ID else 'API キー')

### 0-4. 結果をグラフで可視化する準備

取得済みの回答を横棒グラフにする関数を定義します。このセルでは API を呼びません。`predicate` は返された確率 `p` と補数 `1-p`、`choice` と `score` は選択肢・段階ごとの確率を表示します。スコアと `confidence` は確率分布とは別の値としてグラフの見出しに表示します。

複数質問は質問ごとにグラフを分けます。拒否は「判定が拒否されました」と表示し、確率 0 に置き換えません。日本語フォントが見つからない場合は警告を表示します。

出典: [回答の解釈](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#interpret-answers-and-handle-refusals)、[Matplotlib の横棒グラフ](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.barh.html)

In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_candidates = ['Hiragino Sans', 'Yu Gothic', 'Meiryo', 'Noto Sans CJK JP', 'Noto Sans JP', 'IPAexGothic']
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
japanese_font = next((name for name in font_candidates if name in available_fonts), None)
if japanese_font:
    plt.rcParams['font.family'] = japanese_font
else:
    print('日本語フォントが見つかりません。グラフの日本語が表示されない場合があります。')
plt.rcParams['axes.unicode_minus'] = False

def plot_decision(decision):
    data = decision if isinstance(decision, dict) else decision.model_dump()
    answers = data['answers']
    if not answers:
        raise ValueError('表示する回答がありません。')
    fig, axes = plt.subplots(len(answers), 1, figsize=(9, 3.5 * len(answers)), squeeze=False)
    for ax, answer in zip(axes[:, 0], answers):
        name, kind = answer['name'], answer['type']
        title = f'{name} ({kind})'
        if kind == 'refusal':
            print(f'{name}: 判定が拒否されました。')
            ax.text(0.5, 0.5, f'{name}\n判定が拒否されました', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue
        if kind == 'predicate':
            p = answer['probability']
            labels, values = ['はい', 'いいえ'], [p, 1 - p]
        elif kind in ('choice', 'score'):
            probabilities = answer['probabilities']
            labels = [str(item.get('label') or item['value']) for item in probabilities]
            values = [item['probability'] for item in probabilities]
            selected = answer['choice'] if kind == 'choice' else f"{answer['score']:.3f}"
            title += f'\n{kind}={selected}'
            if answer.get('confidence') is not None:
                title += f", confidence={answer['confidence']:.3f}"
        else:
            raise ValueError(f'未対応の回答形式です: {kind}')
        if not values or not all(math.isfinite(value) and 0 <= value <= 1 for value in values):
            raise ValueError(f'{name}: 確率は 0〜1 の有限値で指定してください。')
        bars = ax.barh(range(len(labels)), values, tick_label=labels, color='#2b6cb0')
        ax.bar_label(bars, fmt='%.3f', padding=3)
        ax.invert_yaxis()
        ax.set_xlim(0, 1.12)
        ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
        ax.set_xlabel('確率')
        ax.set_title(title)
    fig.tight_layout()
    plt.show()

## Part 1: 3 つの質問タイプとレスポンスの構造

### 1-1. predicate: 条件が真である確率

返金を求めているかを質問します。`answers` は配列で、質問の `name` が回答にも入ります。`type` が `predicate` の場合、`probability` は条件が真であるとモデルが推定した確率（0〜1）です。返金を求めているかの推定であり、返金してよいかの判定ではありません。

どの質問でも `type: "refusal"` が返る可能性があります。その場合は判定値を読まず、人間による確認などに回します。以下では回答を JSON で表示し、`type` と値を直接確認します。

出典: [最初の判定](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#make-your-first-decision)、[回答の解釈と拒否](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#interpret-answers-and-handle-refusals)

In [ ]:
Q_REFUND = {
    'type': 'predicate',
    'name': 'refund_requested',
    'instructions': 'お客様は返金を求めていますか？',
}
predicate_decision = client.decisions.create(
    model=DEPLOYMENT,
    input='注文代金が二重に請求されました。余分な請求分を返金してください。',
    questions=[Q_REFUND],
)
print(predicate_decision.model_dump_json(indent=2))
plot_decision(predicate_decision)

### 1-2. choice: 固定の選択肢から 1 つ選ぶ

`choices` に選択肢の `value` と説明を渡します。回答の `choice` は選択した値、`probabilities` は各選択肢の確率です。`confidence` はそれとは別の値として返ります。どれにも当てはまらない入力に備え、`other` も用意します。

出典: [固定の選択肢](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#select-from-fixed-options)

In [ ]:
Q_DEPARTMENT = {
    'type': 'choice',
    'name': 'department',
    'instructions': 'この問い合わせは、どの部署が対応すべきですか？',
    'choices': [
        {'value': 'billing', 'description': '支払い、請求、返金に関する内容。'},
        {'value': 'technical', 'description': '製品の不具合に関する内容。'},
        {'value': 'shipping', 'description': '配送や追跡に関する内容。'},
        {'value': 'other', 'description': '上記に当てはまらない内容。'},
    ],
}
choice_decision = client.decisions.create(
    model=DEPLOYMENT, input='注文代金が二重に請求されました。', questions=[Q_DEPARTMENT]
)
print(choice_decision.model_dump_json(indent=2))
plot_decision(choice_decision)

In [ ]:
choice_decision = client.decisions.create(
    model=DEPLOYMENT, input='まだ商品が届いてないんだけど！？', questions=[Q_DEPARTMENT]
)
print(choice_decision.model_dump_json(indent=2))
plot_decision(choice_decision)

### 1-3. score: 順序のある段階で採点する

`levels` は低い順に並べ、各段階の基準を記述します。段階の添字は 0 から始まり、`score` はその確率加重平均です。この例の範囲は 0〜2 で、整数になるとは限りません。カテゴリを 1 つ選ぶ `choice` とは異なります。

出典: [ルーブリックによる採点](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#score-against-a-rubric)

In [ ]:
Q_SEVERITY = {
    'type': 'score',
    'name': 'severity',
    'instructions': 'この問題の深刻度はどの程度ですか？',
    'levels': [
        {'label': '軽微', 'description': '失われる機能はない。'},
        {'label': '回避策あり', 'description': '作業は失敗するが、別の方法で完了できる。'},
        {'label': '完全停止', 'description': '作業が失敗し、回避策もない。'},
    ],
}
score_decision = client.decisions.create(
    model=DEPLOYMENT,
    input='ファイルをドラッグしてもアップロードできませんが、ファイル選択ボタンからはアップロードできます。',
    questions=[Q_SEVERITY],
)
print(score_decision.model_dump_json(indent=2))
plot_decision(score_decision)

### 1-4. score の計算を手元で再現する

API は呼びません。段階ごとの確率が 0.1・0.7・0.2 なら、`0×0.1 + 1×0.7 + 2×0.2 = 1.1` です。1 に集中した分布と 0・2 に半分ずつ割れた分布は、どちらもスコアが 1 になります。棒グラフを並べ、スコアだけでは区別できない分布の違いを確認します。

出典: [スコアの定義](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#score-against-a-rubric)

In [ ]:
cases = {
    '公式の計算例': [0.1, 0.7, 0.2],
    '1 段階目に集中': [0.0, 1.0, 0.0],
    '0 と 2 に分散': [0.5, 0.0, 0.5],
}
fig, axes = plt.subplots(1, len(cases), figsize=(12, 3.5), sharey=True)
for ax, (label, probabilities) in zip(axes, cases.items()):
    score = sum(i * p for i, p in enumerate(probabilities))
    print(f'{label}: probabilities={probabilities}, score={score:.2f}')
    ax.bar(range(len(probabilities)), probabilities, color='#2b6cb0')
    ax.axvline(score, color='#c05621', linestyle='--', label='スコア')
    ax.set_title(f'{label}\nscore={score:.2f}')
    ax.set_xticks(range(len(probabilities)))
    ax.set_xlabel('段階の添字')
    ax.set_ylim(0, 1.05)
    ax.legend()
axes[0].set_ylabel('確率')
fig.tight_layout()
plt.show()

### 1-5. レスポンス全体と usage を確認する

1-2 の結果を再利用するため、追加の API 呼び出しはありません。`answers` には質問順の回答配列が入り、`usage` にはリクエストの入力・出力・合計トークン数が入ります。個々の判定値だけでなく、レスポンス全体を確認します。

出典: [レスポンスの構造](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#understand-how-decisions-work)

In [ ]:
body = choice_decision.model_dump()
print(json.dumps(body, ensure_ascii=False, indent=2))
print('回答名:', [answer['name'] for answer in body['answers']])
print('トークン使用量:', body['usage'])

### 1-6. REST で直接呼び出す

SDK の代わりに `POST /openai/v1/decisions` を直接呼び出します。リクエスト本文は `model`・`input`・`questions` です。API キーは `api-key`、Microsoft Entra ID は `Authorization: Bearer` ヘッダーを使用します。HTTP エラー時は本文を表示して停止します。

出典: [REST の呼び出し例](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#make-your-first-decision)、[認証](https://learn.microsoft.com/azure/ai-services/openai/how-to/managed-identity)

In [ ]:
import httpx

headers = {'Authorization': 'Bearer ' + token_provider()} if USE_ENTRA_ID else {'api-key': API_KEY}
response = httpx.post(
    ENDPOINT + 'decisions',
    headers=headers,
    json={
        'model': DEPLOYMENT,
        'input': '注文代金が二重に請求されました。余分な請求分を返金してください。',
        'questions': [Q_REFUND],
    },
    timeout=60.0,
)
print('HTTP', response.status_code)
if response.is_error:
    print(response.text)
response.raise_for_status()
rest_result = response.json()
print(json.dumps(rest_result, ensure_ascii=False, indent=2))
plot_decision(rest_result)

## Part 2: 入力のバリエーション

### 2-1. 1 回のリクエストで複数の質問をする

ここで初めて 3 種類の質問を同じリクエストにまとめます。Part 1 で定義した質問を再利用し、共通の `input` に対して部署・深刻度・返金希望を独立して評価します。回答は質問と同じ順序です。ある質問の回答を別の質問への入力として使う処理ではありません。

出典: [複数の質問](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#ask-multiple-questions)

In [ ]:
questions = [Q_DEPARTMENT, Q_SEVERITY, Q_REFUND]
multi_decision = client.decisions.create(
    model=DEPLOYMENT,
    input='二重に請求されたうえ、領収書ページを開くたびにアプリが落ちます。余分な請求分を返金してください。',
    questions=questions,
)
print('質問の順序:', [q['name'] for q in questions])
print('回答の順序:', [answer.name for answer in multi_decision.answers])
print(multi_decision.model_dump_json(indent=2))
plot_decision(multi_decision)

### 2-2. メッセージ形式の入力

`input` は文字列だけでなく、`user` メッセージの配列も受け付けます。`content` を文字列にする形式と、`input_text` の配列にする形式を試します。公式文書に記載された `user` のみを使用し、他のロールの対応は仮定しません。

出典: [入力の形式](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#understand-how-decisions-work)

In [ ]:
message_decision = client.decisions.create(
    model=DEPLOYMENT,
    input=[
        {'role': 'user', 'content': '注文番号 A-1024 の件です。二重に請求されています。'},
        {'role': 'user', 'content': [
            {'type': 'input_text', 'text': '現金の返金ではなく、次回の注文からの値引きで構いません。'},
        ]},
    ],
    questions=[{
        'type': 'predicate',
        'name': 'cash_refund_requested',
        'instructions': '顧客は現金での返金を求めていますか。値引きや交換だけを求めている場合は含めません。',
    }],
)
print(message_decision.model_dump_json(indent=2))
plot_decision(message_decision)

### 2-3. 画像評価の準備

API は呼びません。用意された [正常画像](../product_ok.png) と [破損画像](../product_damaged.png) を再利用し、画像生成処理を省きます。ファイルを表示し、送信用の Base64 データ URL を作る関数を定義します。ローカルのファイル名をそのまま API に渡すことはできません。

両方とも説明用の合成画像であり、実物の検品精度を示すものではありません。

出典: [画像の入力形式](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#evaluate-an-image)

In [ ]:
import base64
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display


def make_product_image(path, damaged):
    img = Image.new('RGB', (480, 360), 'white')
    d = ImageDraw.Draw(img)
    d.rounded_rectangle((90, 70, 390, 300), radius=24, fill=(70, 120, 200),
                        outline=(30, 60, 120), width=6)
    d.rectangle((150, 130, 330, 190), fill=(235, 240, 250))
    if damaged:
        crack = [(250, 70), (235, 120), (270, 160), (225, 215), (260, 260), (245, 300)]
        d.line(crack, fill=(20, 20, 20), width=5)
        d.line([(270, 160), (320, 175), (345, 215)], fill=(20, 20, 20), width=4)
    img.save(path)
    return path


def to_data_url(path, mime='image/png'):
    b64 = base64.b64encode(Path(path).read_bytes()).decode('ascii')
    return f'data:{mime};base64,{b64}'


OK_IMG = make_product_image('product_ok.png', damaged=False)
NG_IMG = make_product_image('product_damaged.png', damaged=True)

In [ ]:
import base64
from IPython.display import Image, display

OK_IMG = 'product_ok.png'
NG_IMG = 'product_damaged.png'

from pathlib import Path

def to_data_url(path):
    return 'data:image/png;base64,' + base64.b64encode(Path(path).read_bytes()).decode('ascii')

display(Image(filename=str(OK_IMG)), Image(filename=str(NG_IMG)))

### 2-4. 画像を評価する

1 枚の画像を `input_image` として渡し、目に見える損傷があるかを `predicate` で評価します。`IMAGE_PATH` を `OK_IMG` に変更すれば正常画像でも試せます。変更後の再実行は追加の API 呼び出しになります。

出典: [画像の評価](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#evaluate-an-image)

In [ ]:
IMAGE_PATH = NG_IMG
image_decision = client.decisions.create(
    model=DEPLOYMENT,
    input=[{'role': 'user', 'content': [
        {'type': 'input_text', 'text': 'この写真の製品を検品してください。'},
        {'type': 'input_image', 'image_url': to_data_url(IMAGE_PATH)},
    ]}],
    questions=[{
        'type': 'predicate',
        'name': 'visible_damage',
        'instructions': '製品にひび割れ、破れ、へこみなどの目に見える損傷がありますか。影や梱包の損傷は無視してください。',
    }],
)
print(image_decision.model_dump_json(indent=2))
plot_decision(image_decision)

### 2-5. 複数の画像にラベルを付けて比較する

画像の直前に `input_text` でラベルを付け、質問でどの画像を指すかを明確にします。1 枚目の損傷、2 枚目の損傷、2 枚目だけに見える損傷を 1 回で評価します。観察できる画像差分の判定であり、破損の原因や実際の発生時点を断定するものではありません。

出典: [画像入力](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#evaluate-an-image)、[複数の質問](https://learn.microsoft.com/azure/foundry/openai/how-to/decisions#ask-multiple-questions)

In [ ]:
comparison_decision = client.decisions.create(
    model=DEPLOYMENT,
    input=[{'role': 'user', 'content': [
        {'type': 'input_text', 'text': '1 枚目（出荷前の写真）:'},
        {'type': 'input_image', 'image_url': to_data_url(OK_IMG)},
        {'type': 'input_text', 'text': '2 枚目（顧客から届いた写真）:'},
        {'type': 'input_image', 'image_url': to_data_url(NG_IMG)},
    ]}],
    questions=[
        {'type': 'predicate', 'name': 'first_damaged',
         'instructions': '1 枚目の写真の製品に、目に見える損傷がありますか。'},
        {'type': 'predicate', 'name': 'second_damaged',
         'instructions': '2 枚目の写真の製品に、目に見える損傷がありますか。'},
        {'type': 'predicate', 'name': 'new_visible_damage',
         'instructions': '1 枚目にはなく、2 枚目にだけ見える製品の損傷がありますか。'},
    ],
)
print(comparison_decision.model_dump_json(indent=2))
plot_decision(comparison_decision)